In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\ashwi\Downloads\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
#Data Understanding
print("Shape:", df.shape)
print(df['sentiment'].value_counts())

Shape: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
#Preprocessing
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

df['clean_text'] = df['review'].apply(preprocess)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashwi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashwi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
#convert to labels
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [5]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
#Feature Engineering
#Bag of words
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [7]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
def evaluate(model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(model.__class__.__name__)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall:", recall_score(y_test, pred))
    print("F1 Score:", f1_score(y_test, pred))
    print("-"*40)

In [10]:
print("BoW Results")
evaluate(LogisticRegression(max_iter=1000), X_train_bow, X_test_bow)
evaluate(MultinomialNB(), X_train_bow, X_test_bow)
evaluate(DecisionTreeClassifier(), X_train_bow, X_test_bow)

BoW Results
LogisticRegression
Accuracy: 0.8756
Precision: 0.8714034057545508
Recall: 0.8835086326652114
F1 Score: 0.8774142688214427
----------------------------------------
MultinomialNB
Accuracy: 0.8486
Precision: 0.8520071899340923
Recall: 0.8465965469339155
F1 Score: 0.8492932510451922
----------------------------------------
DecisionTreeClassifier
Accuracy: 0.712
Precision: 0.7168977295559574
Recall: 0.7080769994046437
F1 Score: 0.7124600638977636
----------------------------------------


In [ ]:
print("TF-IDF Results")

evaluate(LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf)
evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf)
evaluate(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf)

TF-IDF Results
LogisticRegression
Accuracy: 0.8871
Precision: 0.8779969064191802
Recall: 0.9011708672355626
F1 Score: 0.8894329644501028
----------------------------------------
MultinomialNB
Accuracy: 0.8545
Precision: 0.8528948404883813
Recall: 0.8594959317324866
F1 Score: 0.8561826628447168
----------------------------------------


In [ ]:
## Final Insights

- TF-IDF performed better than BoW
- Logistic Regression gave highest accuracy
- Naive Bayes performed well for text classification
- Decision Tree showed lower performance due to overfitting

Best Model: Logistic Regression with TF-IDF